# Build a Simple LLM App with Gradio

![Workshop flow — you are here: Gradio Apps & Streaming](../assets/notebook_flow_diagram.png)

Notebook 1 was about prompts. This one is about **packaging** a prompt so someone else can use it without touching Python.

We will go in small steps: function first, interface second, streaming last.


## Step 1: Same setup as Notebook 1


In [1]:
import sys
from pathlib import Path

# Run this cell first. It finds the project folder and loads your .env file.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

True

In [2]:
from src.llm_gateway import check_available_providers, run_llm, stream_llm

SELECTED_PROVIDER = "auto"
check_available_providers()

{'openai': False,
 'anthropic': False,
 'gemini': False,
 'mistral': False,
 'cohere': False,
 'deepseek': False,
 'groq': False,
 'openrouter': False,
 'ollama': True}

## Step 2: One function, one job

Before any buttons or sliders, we call the model from a plain function.


In [3]:
from src.gradio_apps import generate_learning_support
from src.output_formatting import show_llm_output

result = generate_learning_support(
    user_text="Transformers use attention to relate tokens in a sequence.",
    audience="undergraduate students",
    task_type="Explain simply",
    provider=SELECTED_PROVIDER,
)
show_llm_output(result, title="Learning support preview")


/Users/jeremiah/miniconda3/envs/llm/lib/python3.13/site-packages/gradio/routes.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Learning support preview

## Explain simply

1. Short answer — Transformers are a type of neural network that use attention mechanisms to relate the tokens in a sequence.
2. Key points —
* Transformers use self-attention mechanisms to weigh the importance of different words in a sentence.
* They can be used for tasks such as language translation and text summarization.
3. Example — Imagine you're trying to translate a sentence from English to Spanish. A transformer would analyze the entire sentence, weighing the importance of each word, and then generate the corresponding Spanish sentence.
4. Common misunderstanding — Transformers are not just used for language tasks, they can also be applied to other types of sequential data such as images or audio.
5. Quiz -
a) What is the main function of a transformer?
* To weigh the importance of different words in a sequence
b) What are some examples of tasks that can be performed using transformers?
* Language translation, text summarization

If that worked, the hard part is done. The Gradio app below is mostly wiring.


## Step 3: Launch the learning support app

The interface is intentionally simple:

1. Paste **source text**
2. Choose **what you need** (explain, summarise, quiz, etc.)
3. Click **Generate** — wait 10–30 seconds for the status to say `Done.`

Provider, temperature, and other settings use sensible defaults (`auto` picks Ollama or your API key).

Stop the cell with the stop button when finished.


In [4]:
from src.gradio_apps import build_learning_support_app

# Restart the kernel first if you changed src/gradio_apps.py since launching.
app = build_learning_support_app()
app.launch(share=False, show_error=True)


Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


Try an example or paste your own text. The app keeps the controls minimal so students focus on the prompt pattern behind the scenes (see `src/gradio_apps.py` if you want to add sliders later).


## Step 4: A simple chatbot


In [5]:
from src.gradio_apps import build_chatbot_app

chat = build_chatbot_app(streaming=False)
#chat.launch()

## Step 5: Streaming (why it feels faster)

Streaming prints tokens as they arrive. The total time is similar, but the experience feels quicker.


In [6]:
from IPython.display import Markdown, display

chunks = []
for chunk in stream_llm("Explain overfitting in one paragraph.", provider=SELECTED_PROVIDER):
    chunks.append(chunk)

display(Markdown("".join(chunks)))


 Overfitting occurs when a machine learning model learns the details and noise in the training data to the point that it performs poorly on new, unseen data. This happens when the model is too complex for the amount of training data available, or when the model is trained for too long. Overfitting can lead to poor generalization performance, where the model fails to make accurate predictions on new data. To avoid overfitting, it's important to use a properly sized model and to monitor the model's performance on a validation set during training. 

## Step 6: Streaming chatbot

Same simple chat interface, but the reply streams in word by word. **Restart the kernel** if you already ran an earlier version of this notebook.

In [7]:
from src.gradio_apps import build_chatbot_app

# Restart the kernel first if you changed src/gradio_apps.py since launching.
stream_chat = build_chatbot_app(streaming=True)
stream_chat.launch(share=False, show_error=True)

Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.


## Safety note

The apps warn users not to paste private data. In a real product you would also log errors, limit input length, and document what leaves the machine.

## Mini project ideas

Pick one: student support bot, lecture explainer, abstract helper, code error explainer, assignment feedback helper.

Add your own prompt template, provider choice, Gradio UI, and a short checklist for testing.
